# 01 · Chuẩn bị VNTS + train detector YOLO11n (1 lớp và 52 lớp)

Cần **GPU T4**, bật Internet. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input trên Kaggle:
- dataset `maitam/vietnamese-traffic-signs` (VNTS, 52 lớp đặt tên theo mã QCVN, CC BY-SA 4.0)
- dataset `vn-dashcam-vision-code` (code repo này)

Trên Colab: code clone từ GitHub, VNTS tải bằng `kagglehub` (dataset public, **không cần tài khoản Kaggle**;
có Secrets `KAGGLE_USERNAME`, `KAGGLE_KEY` thì dùng luôn). Kaggle lỗi thì lấy bản mirror trên HF.
Dataset để ở ổ cục bộ `/content/data`, model + kết quả nằm trong `MyDrive/ai-portfolio/vn-dashcam-vision`
để notebook 02-04 dùng lại.

Thời gian trên Colab T4: tải + chuẩn bị dữ liệu ~5-10 phút, train 2 detector ~40 phút (giới hạn cứng bằng `time=`).

Train 2 detector để so sánh:
- `1cls`: mọi biển gộp 1 lớp "sign" -> tầng 1 của pipeline 2 tầng
- `ncls`: 52 lớp -> baseline 1 tầng

Ultralytics là AGPL-3.0: repo public thì ổn, dùng thương mại phải mở mã hoặc mua license.

In [ ]:
import gc, glob, json, os, shutil, sys

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vn-dashcam-vision" if IN_COLAB else os.path.abspath("vn-dashcam-vision")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vn-dashcam-vision {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vn-dashcam-vision" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


# Colab: đọc hàng nghìn ảnh từ Google Drive rất chậm -> dataset để ở ổ cục bộ /content (mất khi ngắt,
# chạy lại cell là có), còn model / kết quả / checkpoint YOLO thì nằm trong WORK trên Drive.
DATA = "/content/data" if IN_COLAB else "datasets"
DS = f"{DATA}/vnts"
os.makedirs(DATA, exist_ok=True)


def get_vnts():
    """VNTS (archive/images, archive/labels, classes.txt, split_dataset/{train,test}_files.txt) -> thư mục gốc."""
    raw = f"{DATA}/vnts_raw"
    if glob.glob(f"{raw}/**/classes.txt", recursive=True):
        return raw
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        v = secret(k)
        if v:
            os.environ[k] = v
    try:  # dataset public: kagglehub tải được cả khi không đăng nhập; có Secrets thì nó tự dùng
        import kagglehub
        return kagglehub.dataset_download("maitam/vietnamese-traffic-signs")
    except Exception as e:
        print("kagglehub lỗi:", repr(e)[:300])
    try:  # mirror HF: cùng 3216 ảnh + nhãn + classes.txt, nhưng KHÔNG có split_dataset/ -> test chia lại, số khác bản Kaggle
        from huggingface_hub import snapshot_download
        print("tải mirror HF thanhhiepvos/vietnam_traffic_sign (6.4k file, vài phút)")
        snapshot_download("thanhhiepvos/vietnam_traffic_sign", repo_type="dataset", local_dir=raw,
                          allow_patterns=["archive/*"])
        return raw
    except Exception as e:
        print("HF lỗi:", repr(e)[:300])
    raise RuntimeError("không tải được VNTS. Tải zip ở kaggle.com/datasets/maitam/vietnamese-traffic-signs "
                       f"rồi giải nén vào {raw}, hoặc thêm Secrets KAGGLE_USERNAME/KAGGLE_KEY")


# VNTS: Kaggle add dataset làm input; Colab tải về ổ cục bộ
if IN_KAGGLE:
    VNTS = [d for d in glob.glob("/kaggle/input/*") if "traffic" in d.lower()][0]
else:
    VNTS = get_vnts()
print(CODE, VNTS)
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if not os.path.exists(f"{DS}/crops/test.npz"):  # chạy lại cell trong cùng runtime thì khỏi làm lại
    !python -m dashcam.prepare_vnts --src {VNTS} --out {DS}
print(json.dumps(json.load(open(f"{DS}/class_stats.json"))["train"], ensure_ascii=False)[:1500])

## Phân bố lớp (lệch rất mạnh)

In [ ]:
import matplotlib.pyplot as plt

st = json.load(open(f"{DS}/class_stats.json"))["train"]
items = sorted(st.items(), key=lambda x: -x[1])
plt.figure(figsize=(16, 4))
plt.bar([k for k, _ in items], [v for _, v in items])
plt.xticks(rotation=90, fontsize=7); plt.yscale("log"); plt.title("số crop train mỗi lớp (log)")
plt.tight_layout(); plt.savefig("class_dist.png", dpi=110)

## Copy-paste cho lớp hiếm (< 50 mẫu train)

In [ ]:
if not glob.glob(f"{DS}/yolo_ncls/images/train/synth_*"):
    !python -m dashcam.synth_rare --data {DS} --rare 50 --per-class 150

## Train YOLO11n
imgsz 640: biển xa chỉ ~15px nên 960 sẽ tốt hơn cho recall, nhưng chạy CPU chậm hơn ~2.25 lần (ước theo diện tích ảnh, chưa đo).
Giữ 640 vì mục tiêu là chạy được trên laptop; ghi lại đánh đổi này trong NOTES.

Colab free chỉ có ~2 nhân CPU nên nút cổ chai là đọc ảnh + mosaic -> cache ảnh đã giải mã ra ổ cục bộ (`cache="disk"`).
Không dùng `cache="ram"`: chạy thật trên Colab (12.7GB RAM) bị crash hết RAM ở bước validate cuối, vì cache train
(~3.6GB) bị nhân lên theo số worker của dataloader. `workers=2` cho khớp 2 nhân CPU.
`time=` (giờ) là giới hạn cứng, ultralytics tự tính lại số epoch + lịch learning rate cho vừa thời gian đó.
Checkpoint ghi thẳng vào Drive: runtime bị ngắt thì chạy lại notebook, cell này train tiếp từ `last.pt`.

In [ ]:
import torch
from ultralytics import YOLO

RUNS = f"{WORK}/runs"  # đường dẫn tuyệt đối: ultralytics 8.4 đưa project tương đối vào runs_dir/detect/...
HOURS = {"1cls": 0.35, "ncls": 0.3}  # tổng ~39 phút trên T4
for name, h in HOURS.items():
    d = f"{RUNS}/det_{name}"
    if os.path.exists(f"{d}/done"):
        print(name, "đã train xong, bỏ qua")
        continue
    last = f"{d}/weights/last.pt"
    if os.path.exists(last) and torch.load(last, map_location="cpu", weights_only=False).get("optimizer") is None:
        # ultralytics xoá optimizer khỏi last.pt khi train xong -> chỉ bị ngắt lúc validate cuối, không cần train lại
        # (resume=True với checkpoint này sẽ âm thầm train mới bằng cấu hình mặc định)
        print(name, "đã train đủ epoch, bỏ qua")
    elif os.path.exists(last):
        YOLO(last).train(resume=True, time=h)
    else:
        YOLO("yolo11n.pt").train(data=f"{DS}/yolo_{name}/data.yaml", imgsz=640, epochs=100, time=h, patience=15,
                                 batch=32, cache="disk", workers=2, close_mosaic=5,
                                 fliplr=0.0,  # KHÔNG lật ngang: cấm rẽ trái <-> cấm rẽ phải
                                 mosaic=1.0, degrees=5, hsv_v=0.5, project=RUNS, name=f"det_{name}", exist_ok=True,
                                 verbose=False)
    open(f"{d}/done", "w").close()
    gc.collect()

In [ ]:
os.makedirs("models", exist_ok=True)
for name in HOURS:
    m = YOLO(f"{RUNS}/det_{name}/weights/best.pt")
    r = m.val(data=f"{DS}/yolo_{name}/data.yaml", split="test", imgsz=640, batch=32, project=RUNS,
              name=f"test_{name}", exist_ok=True, verbose=False)
    print(name, "mAP50", round(r.box.map50, 4), "mAP50-95", round(r.box.map, 4))
    shutil.copy(m.export(format="onnx", imgsz=640, opset=17, simplify=True), f"models/detector_{name}.onnx")

In [ ]:
!cp {DS}/names.json models/ && ls -la models
# giữ crops + class_stats cho notebook 02
!tar czf crops.tgz -C {DATA} vnts/crops vnts/names.json vnts/class_stats.json